In [39]:

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
# from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader=TextLoader("speech.txt")

documents=loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50)

docs = text_splitter.split_documents(documents)


In [40]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of'),
 Document(metadata={'source': 'speech.txt'}, page_content='must be planted upon the tested foundations of political liberty. We have no selfish ends to serve.'),
 Document(metadata={'source': 'speech.txt'}, page_content='liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no'),
 Document(metadata={'source': 'speech.txt'}, page_content='We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation'),
 Document(metadata={'source': 'speech.txt'}, page_content='for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of'),
 Document(metadata={'source': 'speech.txt'}, page_content='we shall freely make. We are but one of the champions of the rights of mankind. We shall be'),
 Document(metadata={'source': 'speech.txt'}, pag

In [41]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = FAISS.from_documents(docs, embeddings)
db

In [45]:
### Querying

query="How does the speaker describe the desired outcome of the war ?"

docs = db.similarity_search(query)

# docs
docs[0].page_content



'war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the'

# As a Retriever

We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other Langchain methods, which largely work with retrievers.


#### Later on when we work with different different models, at that point of time we can not use vector DB directly, we need to convert vector store DB into a retrieve and then we will be able to use with any LLM models.

In [49]:
retriever = db.as_retriever()

retriever.invoke(query)

docs[0].page_content

'war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the'

## Similarity Search with Score

There are some FAISS specific methods. One of them is similarity_search_with_score, which allows us to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance (ManHattan Distance). Therefore a lower score is better.

In [51]:
docs_and_score=db.similarity_search_with_score(query)

docs_and_score

[(Document(id='785c9213-a7fe-4e67-bae7-290a259646d4', metadata={'source': 'speech.txt'}, page_content='war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the'),
  np.float32(318.96667)),
 (Document(id='f3c81c0a-c943-4c87-a25d-afab001fcd0f', metadata={'source': 'speech.txt'}, page_content='fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of'),
  np.float32(319.09445)),
 (Document(id='51eb10db-a071-443b-b0ea-199dde3c337b', metadata={'source': 'speech.txt'}, page_content='wars, civilization itself seeming to be in the balance. But the right is more precious than peace,'),
  np.float32(343.88962)),
 (Document(id='89dfe04c-224c-4786-9fb0-e4e920ce9201', metadata={'source': 'speech.txt'}, page_content='operations as belligerents without passion and ourselves observe with proud punctilio the'),
  np.float32(359.50504))]

In [52]:
embedding_vector= embeddings.embed_query(query)

embedding_vector

[-0.30234602093696594,
 0.2792474925518036,
 -3.6141488552093506,
 -0.9987055063247681,
 2.02024507522583,
 1.4692351818084717,
 -1.0225698947906494,
 0.23453658819198608,
 0.482858806848526,
 -0.17968806624412537,
 -0.062198612838983536,
 0.5966115593910217,
 0.8661834001541138,
 1.6244386434555054,
 2.0875229835510254,
 -0.7807308435440063,
 0.25819095969200134,
 -1.2089486122131348,
 -0.6829149127006531,
 0.7267792820930481,
 -0.9001617431640625,
 -0.3932344317436218,
 0.09138987213373184,
 0.4785754084587097,
 1.3454581499099731,
 0.4109819829463959,
 -0.5241963863372803,
 0.6710165143013,
 -1.0717294216156006,
 -0.5261573195457458,
 0.9152205586433411,
 0.06045959144830704,
 -0.05185677483677864,
 0.2832365036010742,
 -1.5355463027954102,
 -1.9171974658966064,
 0.07939351350069046,
 1.3350889682769775,
 0.35533538460731506,
 -1.427636742591858,
 0.06351285427808762,
 -0.49632102251052856,
 0.1230013519525528,
 -1.5531880855560303,
 1.366951584815979,
 -0.6890202760696411,
 -0.1299

In [ ]:
#Similarityy search by vectors

docs_score = db.similarity_search_by_vector(embedding_vector)

docs_score

[Document(id='785c9213-a7fe-4e67-bae7-290a259646d4', metadata={'source': 'speech.txt'}, page_content='war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the'),
 Document(id='f3c81c0a-c943-4c87-a25d-afab001fcd0f', metadata={'source': 'speech.txt'}, page_content='fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of'),
 Document(id='51eb10db-a071-443b-b0ea-199dde3c337b', metadata={'source': 'speech.txt'}, page_content='wars, civilization itself seeming to be in the balance. But the right is more precious than peace,'),
 Document(id='89dfe04c-224c-4786-9fb0-e4e920ce9201', metadata={'source': 'speech.txt'}, page_content='operations as belligerents without passion and ourselves observe with proud punctilio the')]

In [55]:
## Saving and loading

db.save_local("faiss_index")

In [61]:
new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

docs = new_db.similarity_search(query)

docs

[Document(id='785c9213-a7fe-4e67-bae7-290a259646d4', metadata={'source': 'speech.txt'}, page_content='war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the'),
 Document(id='f3c81c0a-c943-4c87-a25d-afab001fcd0f', metadata={'source': 'speech.txt'}, page_content='fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of'),
 Document(id='51eb10db-a071-443b-b0ea-199dde3c337b', metadata={'source': 'speech.txt'}, page_content='wars, civilization itself seeming to be in the balance. But the right is more precious than peace,'),
 Document(id='89dfe04c-224c-4786-9fb0-e4e920ce9201', metadata={'source': 'speech.txt'}, page_content='operations as belligerents without passion and ourselves observe with proud punctilio the')]